# 06. Results Summary

This is the main reader-facing result notebook for the coursework. It reads saved public CSV/JSON files from `results/` and does not run CLIP, Moment-DETR, or any heavy experiment.


## Main CLIP Sweep on 1,000 Queries

The primary final experiment is the CLIP window/stride/aggregation sweep on a fixed 1,000-query Charades-STA subset.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

def read_json(path):
    with open(ROOT / path, 'r', encoding='utf-8') as f:
        return json.load(f)

clip_1000 = pd.read_csv(ROOT / 'results/charades_sta_sweep_1000/summary.csv')
clip_1000[['config_name', 'window_size', 'stride', 'aggregation', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']]


Conclusion: `clip_w16_s8_mean` is strongest for coarse retrieval at IoU 0.3, while `clip_w8_s4_mean` is strongest for stricter localization at IoU 0.5 and 0.7. This table is the central final result of the coursework.


## Smoothing Experiment on the Same 1,000 Queries

The smoothing ablation reuses the same fixed subset and saved CLIP embeddings. It checks whether moving-average smoothing of frame-level CLIP scores improves localization.


In [ ]:
smoothing = pd.read_csv(ROOT / 'results/charades_sta_smoothing_1000/summary.csv')
smoothing[['config_name', 'smoothing', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU']]


Conclusion: smoothing is an additional ablation. It slightly improves strict localization for the short-window `w8/s4/mean` setup, but it is not a universal improvement and does not replace the main CLIP sweep.


## CLIP vs Moment-DETR 50-Query Feasibility Probe

The comparison below uses a fixed 50-query subset. Moment-DETR is included only as a raw-video feasibility probe, not as a full official benchmark.


In [ ]:
rows = []
for name in ['clip_w8_s4_mean', 'clip_w16_s8_mean', 'clip_w32_s16_mean', 'clip_w16_s8_max']:
    rows.append(read_json(f'results/clip_vs_moment_detr_50/{name}/run_config.json')['summary'])
rows.append({'model': 'Moment-DETR', 'config_name': 'raw_video_checkpoint', **read_json('results/moment_detr_charades_50/metrics.json')})
pd.DataFrame(rows)[['model', 'config_name', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']]


Conclusion: on this small 50-query subset, CLIP provides strong baseline numbers, and Moment-DETR successfully produces raw-video predictions. The comparison is useful for feasibility, but it should not be read as a full Moment-DETR benchmark.


## Final Conclusions

The final public evidence line is Charades-STA + CLIP. The main result is the 1,000-query CLIP sweep; smoothing is a controlled ablation on the same data; Moment-DETR is a limited 50-query feasibility probe. QVHighlights remains part of the project history and limitations discussion, but it is not claimed as a completed benchmark.
